# Get Search Results API

**Endpoint**: `GET /ServicesAPI/API/V1/CMDB/Search`  
**Version**: 03/24/2025  
**Reference**: [Get Search Results API.md](./Get%20Search%20Results%20API.md)

## Overview

Returns a list of device names matching a given search keyword from the NetBrain IE database.

### Key Parameter Rules

| Parameter | Type | Required | Notes |
|-----------|------|----------|-------|
| `keyword` | string | ✅ Yes | Cannot be empty. Max 10 chars (extra chars ignored). |
| `limit` | integer | No | Valid range: 10–100. Default: 50. |
| `index` | integer | No | DB offset for pagination. Default: 0. Cannot be negative. |

**Pagination pattern**: pass `index = last_returned_index + 1` in the next call to avoid duplicates.

---
## 1. Configuration

Update the values below before running any cells.

In [ ]:
# ── Environment ───────────────────────────────────────────────────────────────
NB_URL      = "https://<your-netbrain-host>"   # e.g. https://netbraintech.com
NB_USERNAME = ""                                # NetBrain username
NB_PASSWORD = ""                                # NetBrain password
TENANT_NAME = ""                                # Leave blank if single-tenant
DOMAIN_NAME = ""                                # Leave blank if single-domain

# ── Search defaults ───────────────────────────────────────────────────────────
SEARCH_KEYWORD = "BJ"   # Keyword to search (max 10 chars)
PAGE_LIMIT     = 10     # Results per page (10–100)

---
## 2. Imports & Helper Setup

In [ ]:
import json
import requests
import urllib3
from pprint import pprint

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Base headers reused across all requests
BASE_HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

token = None  # Populated after login

print("✅ Imports loaded.")

---
## 3. Login — Get Authentication Token

In [ ]:
def login(nb_url, username, password, tenant_name="", domain_name=""):
    """Authenticate and return a session token."""
    url = f"{nb_url}/ServicesAPI/API/V1/Session"
    payload = {
        "username": username,
        "password": password
    }
    if tenant_name:
        payload["tenantName"] = tenant_name
    if domain_name:
        payload["domainName"] = domain_name

    resp = requests.post(url, json=payload, headers=BASE_HEADERS, verify=False)
    resp.raise_for_status()
    data = resp.json()
    if data.get("statusCode") == 790200:
        print(f"✅ Login successful. Token: {data['token'][:12]}...")
        return data["token"]
    else:
        raise RuntimeError(f"Login failed: {data}")

token = login(NB_URL, NB_USERNAME, NB_PASSWORD, TENANT_NAME, DOMAIN_NAME)

---
## 4. Core API Call — Get Search Results

In [ ]:
def get_search_results(nb_url, token, keyword, limit=50, index=0):
    """
    GET /ServicesAPI/API/V1/CMDB/Search

    Args:
        keyword (str): Search keyword. Max 10 chars.
        limit   (int): Results per page. Range: 10–100. Default: 50.
        index   (int): DB offset (for pagination). Default: 0.

    Returns:
        dict: Parsed JSON response.
    """
    url = f"{nb_url}/ServicesAPI/API/V1/CMDB/Search"
    headers = {**BASE_HEADERS, "token": token}
    params = {
        "keyword": keyword,
        "limit":   limit,
        "index":   index
    }

    resp = requests.get(url, headers=headers, params=params, verify=False)
    return resp.json()


# ── Basic call ────────────────────────────────────────────────────────────────
print(f"🔍 Searching for: '{SEARCH_KEYWORD}' (limit={PAGE_LIMIT}, index=0)\n")
result = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=PAGE_LIMIT, index=0)
pprint(result)

---
## 5. Parse & Display Results

In [ ]:
if result.get("statusCode") == 790200:
    devices = result.get("searchResult", {}).get("devices", [])
    print(f"✅ Success — {len(devices)} device(s) returned:\n")
    for i, d in enumerate(devices, start=1):
        print(f"  {i:>3}. {d['deviceName']}")
else:
    print(f"❌ Error {result.get('statusCode')}: {result.get('statusDescription')}")

---
## 6. Pagination — Fetch All Results

Pass `index = last_call_last_index + 1` on each subsequent call to page through all results.

In [ ]:
def get_all_search_results(nb_url, token, keyword, limit=50):
    """
    Paginate through all search results for a keyword.

    Returns:
        list: All device name strings across all pages.
    """
    all_devices = []
    index = 0
    page = 1

    while True:
        print(f"  📄 Page {page} — index={index} ...")
        resp = get_search_results(nb_url, token, keyword=keyword, limit=limit, index=index)

        if resp.get("statusCode") != 790200:
            print(f"  ❌ Error on page {page}: {resp.get('statusDescription')}")
            break

        batch = resp.get("searchResult", {}).get("devices", [])
        if not batch:
            print("  ✅ No more results.")
            break

        all_devices.extend([d["deviceName"] for d in batch])
        print(f"     → {len(batch)} device(s) received (total so far: {len(all_devices)})")

        if len(batch) < limit:
            # Last page — fewer results than requested means we've reached the end
            break

        index += len(batch)
        page  += 1

    return all_devices


print(f"📦 Fetching ALL results for keyword='{SEARCH_KEYWORD}' ...\n")
all_devices = get_all_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=PAGE_LIMIT)

print(f"\n🎯 Total devices found: {len(all_devices)}")
for name in all_devices:
    print(f"  • {name}")

---
## 7. Edge Case & Error Tests

Run each cell independently to verify API behavior on invalid inputs.

In [ ]:
# ── Test 1: Keyword longer than 10 chars (should be silently truncated) ───────
print("[Test 1] Keyword > 10 chars — expect: truncated to 10, results based on first 10 chars")
resp = get_search_results(NB_URL, token, keyword="ABCDEFGHIJK", limit=10, index=0)  # 11 chars
pprint(resp)
print()

In [ ]:
# ── Test 2: Empty keyword (should return error) ───────────────────────────────
print("[Test 2] Empty keyword — expect: error (keyword cannot be empty)")
resp = get_search_results(NB_URL, token, keyword="", limit=10, index=0)
pprint(resp)
print()

In [ ]:
# ── Test 3: limit < 10 (below valid range) ────────────────────────────────────
print("[Test 3] limit=5 (below minimum 10) — expect: error 791001")
resp = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=5, index=0)
pprint(resp)
print()

In [ ]:
# ── Test 4: limit > 100 (above valid range) ───────────────────────────────────
print("[Test 4] limit=200 (above maximum 100) — expect: error")
resp = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=200, index=0)
pprint(resp)
print()

In [ ]:
# ── Test 5: Negative limit ────────────────────────────────────────────────────
print("[Test 5] limit=-1 — expect: error 791001 'limit cannot be negative'")
resp = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=-1, index=0)
pprint(resp)
print()

In [ ]:
# ── Test 6: Negative index ────────────────────────────────────────────────────
print("[Test 6] index=-1 — expect: error 791001 'index cannot be negative'")
resp = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=10, index=-1)
pprint(resp)
print()

In [ ]:
# ── Test 7: index beyond total result count (expect empty list, 790200) ───────
print("[Test 7] index=99999 (beyond all results) — expect: 790200 with empty devices list")
resp = get_search_results(NB_URL, token, keyword=SEARCH_KEYWORD, limit=10, index=99999)
pprint(resp)
print()

In [ ]:
# ── Test 8: Keyword that matches nothing ──────────────────────────────────────
print("[Test 8] Keyword with no matches — expect: 790200 with empty devices list")
resp = get_search_results(NB_URL, token, keyword="ZZZZZZZZZ", limit=10, index=0)
pprint(resp)
print()

In [ ]:
# ── Test 9: No limit/index (use API defaults: 50 results from index 0) ────────
print("[Test 9] No limit or index provided — expect: up to 50 results from index 0")
url     = f"{NB_URL}/ServicesAPI/API/V1/CMDB/Search"
headers = {**BASE_HEADERS, "token": token}
resp    = requests.get(url, headers=headers, params={"keyword": SEARCH_KEYWORD}, verify=False)
pprint(resp.json())
print()

---
## 8. Logout

In [ ]:
def logout(nb_url, token):
    url     = f"{nb_url}/ServicesAPI/API/V1/Session"
    headers = {**BASE_HEADERS, "token": token}
    resp    = requests.delete(url, headers=headers, verify=False)
    data    = resp.json()
    if data.get("statusCode") == 790200:
        print("✅ Logged out successfully.")
    else:
        print(f"⚠️  Logout response: {data}")

logout(NB_URL, token)